# Day 5 | Streamlit 대시보드 제작 및 배포
## Korean Stock Undervaluation Analysis — Dashboard

**목표:**
- Streamlit 6탭 대시보드 제작
- GitHub 업로드 + Streamlit Cloud 배포

**대시보드 구성:**
```
Tab 1: 저평가 스크리닝
       - 저평가 TOP 10 (세로 막대그래프)
       - 섹터별 저평가 TOP 3 (카드 형태)
       - 시그널 필터 테이블
       - 전체 종목 버블차트

Tab 2: 괴리율 분석
       - 기본괴리율 vs 조정괴리율 비교
       - 시그널 변화 히트맵
       - 섹터별 평균 조정괴리율

Tab 3: YoY 성장률
       - 성장률 분포 히스토그램
       - YoY vs 괴리율 산점도

Tab 4: 종목 상세
       - 재무데이터 카드
       - 평균 조정괴리율 게이지

Tab 5: 금융 섹터
       - ROE 기반 분석

Tab 6: 업데이트 내역
```

**배포 URL:** https://stock-analysis-dongwooyun.streamlit.app/


## 0. 환경 세팅

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os
import subprocess

BASE_PATH = '/content/drive/MyDrive/stock-analysis'
APP_PATH  = f'{BASE_PATH}/streamlit_app'

# 데이터 확인
df_v2    = pd.read_csv(f'{BASE_PATH}/data/output/gap_v2.csv', dtype={'Code': str})
df_final = pd.read_csv(f'{BASE_PATH}/data/output/gap_v2_final.csv', dtype={'Code': str})

print(f"✅ gap_v2.csv    : {len(df_v2)}개")
print(f"✅ gap_v2_final  : {len(df_final)}개")
print(f"   강력매수       : {(df_v2['signal_v2']=='강력매수').sum()}개")
print(f"   최종 저평가    : {len(df_final)}개")


## 1. Streamlit 앱 파일 구조

In [ ]:
# streamlit_app/ 구조 확인
print("📁 streamlit_app/ 구조:")
for root, dirs, files in os.walk(APP_PATH):
    level = root.replace(APP_PATH, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        print(f'{indent}  {f}')


## 2. 주요 시각화 로직 요약

In [ ]:
# ── 괴리율 조정 공식 ─────────────────────────────────
# 고평가(gap > 0): YoY 성장률만큼 완화
# 저평가(gap < 0): 유지

ADJUST_FACTOR = 0.5  # 조정 강도

def apply_growth_adjustment(gap, yoy_pct):
    if pd.isna(gap):          return None
    if pd.isna(yoy_pct) or yoy_pct <= 0: return round(gap, 4)
    yoy_decimal = min(yoy_pct, 100.0) / 100.0
    if gap > 0:
        return round(gap - (gap * yoy_decimal * ADJUST_FACTOR), 4)
    return round(gap, 4)

# ── 시그널 분류 기준 ──────────────────────────────────
# (시장 조정괴리율 + 섹터 조정괴리율) / 2 평균 기준
SIGNAL_THRESHOLDS = {
    '강력매수': -0.30,   # 평균 -30% 이하
    '매수':     -0.15,   # 평균 -15% 이하
    '중립':      0.15,   # -15% ~ +15%
    '매도':      0.30,   # +15% 이상
    '강력매도':  9999,   # +30% 이상
}
# ⚠️ 기준값(±15%, ±30%)은 임의 설정 → 백테스팅 검증 필요

print("✅ 주요 로직 확인 완료")
print(f"   조정 강도(ADJUST_FACTOR): {ADJUST_FACTOR}")
print(f"   YoY 캡: 100%")
print(f"   시그널 기준: ±15%, ±30%")


## 3. GitHub 업로드 + Streamlit Cloud 배포

In [ ]:
# GitHub 업로드
GITHUB_TOKEN = "여기에_토큰_입력"
GITHUB_USER  = "DongWooYun"
REPO_NAME    = "stock-analysis"

os.chdir(APP_PATH)

cmds = [
    "git add .",
    'git commit -m "Day 5: Streamlit dashboard v1.0"',
    f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git',
    "git push origin main"
]

for cmd in cmds:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"{'✅' if result.returncode==0 else '❌'} {cmd.split()[1]}")

print(f"\n🎉 배포 완료!")
print(f"   GitHub  : https://github.com/{GITHUB_USER}/{REPO_NAME}")
print(f"   Streamlit: https://stock-analysis-dongwooyun.streamlit.app/")


## 4. 대시보드 주요 결과 요약

In [ ]:
# 최종 결과 요약 출력
print("=" * 55)
print("📊 프로젝트 최종 결과 요약")
print("=" * 55)
print(f"분석 대상   : {len(df_v2)}개 (KOSPI200 + KOSDAQ100)")
print(f"YoY 수집    : {df_v2['yoy_growth_pct'].notna().sum()}개")
print(f"최종 저평가 : {len(df_final)}개")
print(f"강력매수    : {(df_v2['signal_v2']=='강력매수').sum()}개")
print()
print("── 최종 저평가 TOP 10 ──")
top10 = df_final.sort_values('gap_market_v2').head(10)
print(top10[['Name', 'gap_market_v2', 'gap_sector_v2',
             'yoy_growth_pct', 'signal_v2']].to_string(index=False))
print()
print("── signal_v2 분포 ──")
print(df_v2['signal_v2'].value_counts().to_string())
